In [ ]:
# # Install dependencies if needed
# !pip install requests matplotlib -q
# curl http://0.0.0.0:8000/metrics

In [ ]:
import requests
import re
import matplotlib.pyplot as plt
from datetime import datetime

# vLLM metrics endpoint
VLLM_METRICS_URL = "http://0.0.0.0:8000/metrics"

response = requests.get(VLLM_METRICS_URL)
metrics_text = response.text.splitlines()

print("Fetched", len(metrics_text), "lines of metrics.")

In [ ]:
# Parse TTFT histogram buckets
ttft_buckets = {}
ttft_pattern = re.compile(r'vllm:time_to_first_token_seconds_bucket\{.*le="([0-9\.]+)".*\} ([0-9\.]+)')

for line in metrics_text:
    match = ttft_pattern.match(line)
    if match:
        le = float(match.group(1))
        count = float(match.group(2))
        ttft_buckets[le] = count

# Sort by bucket size
ttft_buckets = dict(sorted(ttft_buckets.items()))

# Print the parsed histogram
for k, v in ttft_buckets.items():
    print(f"<= {k} sec: {v} requests")

In [ ]:
# Plot TTFT histogram
plt.figure(figsize=(10, 5))
plt.bar(ttft_buckets.keys(), ttft_buckets.values(), width=0.1, align='edge')
plt.xlabel("Time to First Token (seconds)")
plt.ylabel("Cumulative Request Count")
plt.title("vLLM Time to First Token Histogram")
plt.grid(True)
plt.show()

In [ ]:
# Parse GPU cache usage (latest snapshot only)
gpu_usage_values = []
gpu_pattern = re.compile(r'vllm:gpu_cache_usage_perc(?:\{.*\})? ([0-9\.]+)')

for line in metrics_text:
    match = gpu_pattern.match(line)
    if match:
        usage = float(match.group(1)) * 100  # Convert to percentage
        gpu_usage_values.append(usage)

if gpu_usage_values:
    plt.figure(figsize=(6, 4))
    plt.bar(["GPU KV Cache Usage"], [gpu_usage_values[0]])
    plt.ylim(0, 100)
    plt.ylabel("Usage (%)")
    plt.title("Current GPU KV Cache Usage")
    plt.grid(axis='y')
    plt.show()
else:
    print("No GPU usage data found.")

In [ ]:
# Parse prefix cache counters
prefix_cache_hit = 0
prefix_cache_query = 0

for line in metrics_text:
    if "vllm:prefix_cache_hit_total" in line:
        prefix_cache_hit = float(line.split()[-1])
        print("hi")
    elif "vllm:prefix_cache_query_total" in line:
        prefix_cache_query = float(line.split()[-1])
        print("yo")

# Calculate hit rate
prefix_hit_rate = (
    prefix_cache_hit / prefix_cache_query
    if prefix_cache_query > 0 else None
)

In [ ]:
# Plot prefix cache hit rate
if prefix_hit_rate is not None:
    plt.figure(figsize=(5, 3))
    plt.bar(["Prefix Cache Hit Rate"], [prefix_hit_rate * 100])
    plt.ylabel("Hit Rate (%)")
    plt.ylim(0, 100)
    plt.title("Prefix Cache Hit Rate")
    plt.grid(axis='y')
    plt.show()
else:
    print("Prefix cache hit rate data not found or no queries made yet.")